# 02 - Train the Detection Head (RSNA Pneumonia)

Trains DenseNet-121 encoder + a Tiny-YOLOv2 style detection head, using the YOLO loss (coordinate + objectness + no-object + class terms).

In [ ]:
import os
import sys

REPO_NAME = "multi-task-chest-xray-analysis-system"
PROJECT_DIR = f"/kaggle/working/{REPO_NAME}"

assert os.path.exists(PROJECT_DIR), f"{PROJECT_DIR} does not exist!"

sys.path.insert(0, PROJECT_DIR)

print("Project directory:", PROJECT_DIR)

import torch
from torch.utils.data import DataLoader
from src import config, engine
from src.models.mtl_model import MultiCheXNet
from src.datasets.rsna import (RSNADataset, load_rsna_dataframe, get_default_train_augmentation,
                                train_val_split, rsna_collate_fn)
from src.utils.visualize import show_image_boxes, show_training_curves
from src.utils.bbox_utils import decode_predictions

print("device:", config.DEVICE)


## 1. Point these at your downloaded data (see notebook 00)

In [ ]:
SEG_CSV_PATH = "/kaggle/input/datasets/jesperdramsch/siim-acr-pneumothorax-segmentation-data/train-rle.csv"
SEG_IMG_PATH = "/kaggle/input/datasets/jesperdramsch/siim-acr-pneumothorax-segmentation-data"
DET_CSV_PATH = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv"
DET_IMG_PATH = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_images"


df = load_rsna_dataframe(DET_CSV_PATH)
print(len(df), "rows,", df['patientId'].nunique(), "unique images")
df.head()


In [ ]:
train_ids, val_ids = train_val_split(df, val_fraction=0.15)
print(len(train_ids), "train images,", len(val_ids), "val images")

train_ds = RSNADataset(df, train_ids, DET_IMG_PATH, augmentation=get_default_train_augmentation())
val_ds = RSNADataset(df, val_ids, DET_IMG_PATH, augmentation=None)

BATCH_SIZE = 16
# num_workers=0 avoids the "AssertionError: can only test a child process"
# noise from DataLoader worker cleanup under Kaggle/Jupyter + CUDA. Cosmetic
# only, negligible throughput cost at this batch/image size.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                          drop_last=True, collate_fn=rsna_collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
                        collate_fn=rsna_collate_fn)


## 2. Peek at a training sample with its ground-truth boxes

In [ ]:
img, target, label, boxes = train_ds[next(i for i in range(len(train_ds)) if len(train_ds[i][3])>0)]
show_image_boxes(img.permute(1,2,0).numpy(), boxes, title=f"class={label}")


## 3. Build the model (detection head only) and train

In [ ]:
model = MultiCheXNet(pretrained_encoder=True, add_classifier=False,
                     add_detector=True, add_segmenter=False).to(config.DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scaler = torch.amp.GradScaler('cuda', enabled=(config.DEVICE == "cuda"))
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2)

# Previous run: val_loss bottomed out at epoch 5 (0.34) then rose to 0.41 by
# epoch 15 - classic overfitting. Early stopping (patience=5) stops training
# once it stops improving instead of training 10 more wasted/overfit epochs;
# the checkpoint at the true best epoch is what gets kept either way.
EPOCHS = 15
history = engine.fit(
    engine.train_one_epoch_detector, engine.evaluate_detector,
    model, train_loader, val_loader, optimizer, EPOCHS,
    device=config.DEVICE, checkpoint_path="/kaggle/working/multi-task-chest-xray-analysis-system/detector_best.pt",
    monitor="loss", mode="min", scaler=scaler,
    scheduler=scheduler, patience=5,
)


## 4. Training curves & qualitative results

In [ ]:
show_training_curves({k: v for k, v in history.items() if 'loss' in k})


In [ ]:
model.load("/kaggle/working/multi-task-chest-xray-analysis-system/detector_best.pt", map_location=config.DEVICE)
model.eval()

idx = next(i for i in range(len(val_ds)) if len(val_ds[i][3])>0)
img, target, label, gt_boxes = val_ds[idx]
with torch.no_grad():
    det_out = model(img.unsqueeze(0).to(config.DEVICE))["det"][0]
preds = decode_predictions(det_out, conf_threshold=0.3)

show_image_boxes(img.permute(1,2,0).numpy(), gt_boxes, title="ground truth")
show_image_boxes(img.permute(1,2,0).numpy(),
                 [p["box"] for p in preds], [p["score"] for p in preds],
                 title="prediction")


Checkpoint saved to `/content/detector_best.pt`. Reused in `04_Train_MTL_Joint.ipynb`.